<img src="https://datasciencedegree.wisconsin.edu/wp-content/themes/data-gulp/images/logo.svg" width="300">
<img src="https://pandas.pydata.org/static/img/pandas.svg" width="200">

 # <center>Notebook - Lesson 6a - Filtering Data Frames </center>

It's really common to select data matching some condition using `pandas`.  This notebook teaches some of the pattern:

```
df[df[col]==val]
```

and more generally,

```
df[ logical condition ]
```

A book section is recommended 🐼:
* *Python for Data Analysis*, third edition: Section 5.2, subsection "Indexing, Selection, and Filtering", pages 142-147

---

## Imports up top

We'll start by importing NumPy and pandas:

In [ ]:
import pandas as pd

##  Load data from file

I'm going to use the 2010-2019 weather data from the Chippewa Valley Regional Airport for this notebook.

If you get a message you get when running the following, set `low_memory = False` and run again.

In [ ]:
weather_2010 = pd.read_csv('2010.csv')

In [ ]:
# run this cell to see a nice table of data.
weather_2010

Notes:

* This data frame has 124 columns of data.
* There are 124039 rows of data.  Some rows are just before midnight, others are scattered throughout the day. 

In [ ]:
print(list(weather_2010.columns))

## Selecting data: where a column is a specific value

Let's find the rows in the `weather_2010` data frame that have a specific "REPORT_TYPE".  I'll arbitrarily choose one.

First, let's find out what types are available using the `.unique()` function:

In [ ]:
weather_2010['REPORT_TYPE'].unique()

Let's select the "SOD  " report type.  I find it annoying to have the spaces, but we'll just accept them for now.

We need to write a condition that produces True/False values, and feed that to the `[]` accessor operator for the data frame, in the canonical pattern for selecting data:

```
df[ boolean Series ]
```

In [ ]:
# this produces True/False (booleans)
weather_2010['REPORT_TYPE']=='SOD  '

In [ ]:
# next, stuff that inside [] to select just those rows with a True.  
# Note that this produces a temporaray data frame, and does not modify the original data

weather_2010[ weather_2010['REPORT_TYPE']=='SOD  ' ]
#             ^^  conditional statement        ^^
#          ^^  wrapped in square braces            ^^
# ^^ selects rows from the data frame where condition is True

The selected data using `weather_2010[ weather_2010['REPORT_TYPE']=='SOD  ' ]` appears to give us Sunrise and Sunset data:

In [ ]:
weather_2010[ weather_2010['REPORT_TYPE']=='SOD  ' ][ ['DATE', 'Sunrise', 'Sunset'] ]
#                                                     ^^  a list of column names ^^
# ^^^  a temporary data frame of matching rows
#                                                   ^^accessing into temp df        ^^

Let's capture the result of this temporary, just with the sunrise and sunset data, and play a bit further:

In [ ]:
sunrise_sunset = weather_2010[ weather_2010['REPORT_TYPE']=='SOD  ' ][ ['DATE','Sunrise', 'Sunset', 'DailyMinimumDryBulbTemperature', 'DailyMaximumDryBulbTemperature'] ]

Q: What is the earliest sunrise?

In [ ]:
sunrise_sunset['Sunrise'].min()

Q: On what day is the earliest sunrise?

In [ ]:
row_of_first_earliest = sunrise_sunset['Sunrise'].argmin()
#                                                 ^^ argmin gets the row index of the (first) min

sunrise_sunset['DATE'].iloc[row_of_first_earliest]
#                           ^^ use that row index to get the row
#             ^^ and get the date
# ^^ from the data frame

Q: How many days in this data frame have this same sunrise time? (4:19am, but encoded weird)

In [ ]:
sum(sunrise_sunset['Sunrise']==sunrise_sunset['Sunrise'].min())
#    ^^ produce a Series of True/False, where the sunrise == min 
# ^^ then take sum -- True is 1, False is 0, so summing is counting.

Q: On what day is the latest sunset?

In [ ]:
row_of_first_latest = sunrise_sunset['Sunset'].argmax()
sunrise_sunset['DATE'].iloc[row_of_first_latest]
#             ^^^^ use the date column
#                          ^^ use the index we stored of the (first) latest sunset

Q: What was the high temperature on that date?

In [ ]:
sunrise_sunset['DailyMaximumDryBulbTemperature'].iloc[row_of_first_latest]

I find it relieving that the latest sunset occurs on the summer solstice.  Satisfied sigh.

Now your turn.

🧠 On what date was the (first) earliest **sunset**?

In [ ]:
# 🧠 your code here


🧠 What was the *minimum* dry bulb tempurature on the day of the (first) earliest sunset?

In [ ]:
# 🧠 your code here


## Selecting data: using "greater than"

Let's select the rows of the data frame where the `DailyMaximumDryBulbTemperature` is greater than 80.


The pattern is:

```
df[ df[colname] > 80]
```

In [ ]:
weather_2010[ weather_2010['DailyMaximumDryBulbTemperature']>80 ]
#             ^^ conditional producing booleans             ^^^
#           ^^ [] to access into the complete data frame        ^^
# produces a temporary variable with the requested data

🧠 How many days in this decade of data had a "Daily maximum dry bulb temperature" less than 32 degrees???

(On how many days was the high temperature less than freezing?)

In [ ]:
# 🧠 Your code here


## Selecting data: using a helper lambda: Observations in the month of December

Let's choose to find the data from the month of December. 

The pattern to select is 

```
df[ condition on df ]
```

We need to find a condition on a column of the data frame that gives us "the month is December".  I'll guide you through it.  

### Getting a single entry to play with

I like to start figuring out how to write my selector by getting a single value in memory.  We get data from a DataFrame using the `[]` accessor operator.

In [ ]:
d = weather_2010['DATE'][0]
# d for date.
# [0] to get the top one.

# now we'll have variable `d` to play with.

In [ ]:
# try this, it will fail:  (until you run the next cell, after which it will succeed)
d.month

You should have gotten an error:

`AttributeError: 'str' object has no attribute 'month'`

Indeed, strings don't have months.  I'll convert the column to a datetime column so that I have datetimes instead of strings, cuz then I don't have to write silly string parsing code to extract the month form the string.

In [ ]:
# do the conversion
weather_2010['DATE'] = pd.to_datetime(weather_2010['DATE'])
#                       ^^ conversion to datetime
#  ^^^ overwrite the same column with new data

In [ ]:
# get the 0th entry again.  this time, it's a datetime.
d = weather_2010['DATE'][0]

In [ ]:
# this time, succeeds.
d.month

We can see that the month is an integer from 1 to 12, with 1 being January and 12 being December.  

Turning back to our objective, "get the data from December", we need to write an expression that tests if the month is 12.  Again, the pattern to select data in a pandas data frame is 

```
df[ condition on df ]
```

In [ ]:
# I *want* to write this, but it won't work
weather_2010[weather_2010['DATE'].month == 12]

I can't just go `.month`, unfortunately.  I need to write a little function (a lambda) to get the month and test if it's equal to 12.  I'm going to use the `map` function to have pandas call the function for every element in the data frame.

In [ ]:
weather_2010[weather_2010['DATE'].map(lambda d: d.month) == 12]
#                                     ^^^  gets the month
#                                ^^^ map calls lambda for every element
#                                                        ^^^ tests if equal to 12
#            ^^^ uses a column of True and False values to select the desired rows from the data frame.

Let's save the December data into a variable:

In [ ]:
december_data = weather_2010[weather_2010['DATE'].map(lambda d: d.month) == 12]

🧠 What was the minimum "daily maximum dry bulb tempurature" in all December days from 2010 to 2019?

In [ ]:
# 🧠 your code here


---

Credits: Written by silviana amethyst, phd.  fall 2023.